In [1]:
from pathlib import Path
import os
import sys

from matplotlib import pyplot as plt
import seaborn as sns
import scipy.linalg
import mne
import numpy as np

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)

os.chdir(project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from modules.mvar_model import mvar_optimized

%matplotlib qt

In [2]:
epochs = mne.read_epochs("data/processed/V2/epochs-mvar.fif", preload=True)

Reading c:\Repositorios\msc-eeg-tms-pipeline\data\processed\V2\epochs-mvar.fif ...
    Found the data of interest:
        t =   -2000.00 ...     -50.00 ms
        0 CTF compensation matrices available
Not setting metadata
97 matching events found
No baseline correction applied
0 projection items activated


C:\Users\marci\AppData\Local\Temp\ipykernel_46536\2278889652.py:1: RuntimeWarning: This filename (data/processed/V2/epochs-mvar.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs("data/processed/V2/epochs-mvar.fif", preload=True)


In [3]:
epochs_right, epochs_left, epochs_both = epochs['tms_pulse_task_right'], epochs['tms_pulse_task_left'], epochs['tms_pulse_task_bilateral']

In [9]:
mvar_right = mvar_optimized(epochs_left)

In [10]:
mvar_right.find_order_delta(min_p=9, deltas=[0.01, 0.1])

p= 9, delta=0.010, BIC=-47.2088
p=10, delta=0.010, BIC=-47.0846
p=11, delta=0.010, BIC=-46.9567
p=12, delta=0.010, BIC=-46.8206
p=13, delta=0.010, BIC=-46.6974
p=14, delta=0.010, BIC=-46.5610
p=15, delta=0.010, BIC=-46.4387
p=16, delta=0.010, BIC=-46.3021
p=17, delta=0.010, BIC=-46.1664
p=18, delta=0.010, BIC=-46.0403
p=19, delta=0.010, BIC=-45.9100
p=20, delta=0.010, BIC=-45.7730
p=21, delta=0.010, BIC=-45.6398
p=22, delta=0.010, BIC=-45.5040
p=23, delta=0.010, BIC=-45.3748
p=24, delta=0.010, BIC=-45.2412
p=25, delta=0.010, BIC=-45.1091
p= 9, delta=0.100, BIC=-47.1845
p=10, delta=0.100, BIC=-47.0606
p=11, delta=0.100, BIC=-46.9286
p=12, delta=0.100, BIC=-46.8012
p=13, delta=0.100, BIC=-46.6663
p=14, delta=0.100, BIC=-46.5321
p=15, delta=0.100, BIC=-46.4094
p=16, delta=0.100, BIC=-46.2923
p=17, delta=0.100, BIC=-46.1512
p=18, delta=0.100, BIC=-46.0244
p=19, delta=0.100, BIC=-45.8844
p=20, delta=0.100, BIC=-45.7706
p=21, delta=0.100, BIC=-45.6289
p=22, delta=0.100, BIC=-45.5023
p=23, de

[{'order': 9, 'delta': 0.01, 'bic': np.float64(-47.20880732097359)},
 {'order': 9, 'delta': 0.1, 'bic': np.float64(-47.18451920565308)},
 {'order': 10, 'delta': 0.01, 'bic': np.float64(-47.084555894037145)},
 {'order': 10, 'delta': 0.1, 'bic': np.float64(-47.060644876228)},
 {'order': 11, 'delta': 0.01, 'bic': np.float64(-46.95669955190013)}]

In [12]:
mvar_right.fit_model(35, 0.01)

Model fitted with optimal order and delta.
The fitted VAR model is stable.
Whiteness Test p-value: 0.0000

Biggest eigenvector: 0.9830 (Must be < 1 for stability)


In [7]:
pdc_result, pdc_matrix_eeg = mvar_right.get_connectivity()

In [8]:
plt.figure(figsize=(12, 10))

sns.heatmap(pdc_matrix_eeg, 
            xticklabels=epochs_right.ch_names, 
            yticklabels=epochs_right.ch_names,
            cmap="Reds",
            annot=False,
            cbar_kws={'label': 'PDC Médio (8-30 Hz)'})

plt.title("Matriz de Conectividade PDC entre CANAIS DE EEG (8-30 Hz)\nLeitura: Coluna (Origem) → Linha (Destino)")
plt.xlabel("Canal de Origem (Transmissor)")
plt.ylabel("Canal de Destino (Receptor)")
plt.tight_layout()
plt.show()

In [9]:
validated_pdc = mvar_right.validation_connectivity(pdc_result, n_surrogates=200)

Performing surrogate connectivity analysis with 300 surrogates...


In [10]:
plt.figure(figsize=(12, 10))

sns.heatmap(validated_pdc[:,:,20], 
            xticklabels=epochs_right.ch_names, 
            yticklabels=epochs_right.ch_names,
            cmap="Reds",
            annot=False,
            cbar_kws={'label': 'PDC Médio (8-30 Hz)'})

plt.title("Matriz de Conectividade PDC entre CANAIS DE EEG (8-30 Hz)\nLeitura: Coluna (Origem) → Linha (Destino)")
plt.xlabel("Canal de Origem (Transmissor)")
plt.ylabel("Canal de Destino (Receptor)")
plt.tight_layout()
plt.show()

In [ ]:
bootstrap_pdc, ci_lower, ci_upper = mvar_right.bootstrap_connectivity(measure='PDC', n_bootstraps=300)